# 01 Datasets and preprocessing

Loads all five classification datasets with the same edge check, size filter and preprocessing that the other
notebooks use (the preprocessed inputs are cached in `results/cache`, so the later notebooks start faster).
No network is trained here.

| Dataset | Camera | Pixel size after resampling | Group used in the splits |
|---|---|---|---|
| z-stack (static) | Andor Zyla | 0.65 µm/px | stack, acquisition day |
| Andor Zyla flow | Andor Zyla | 0.65 µm/px | recording |
| static Phantom | Phantom TMX | 0.925 µm/px | raw image (BF and Ph2 together) |
| Phantom flow | Phantom TMX | 0.925 µm/px | recording |
| APD | Phantom TMX | 0.925 µm/px | video |

Thesis: Methods, Sections "Datasets" (Table `tab:dataset-overview`) and "Preprocessing" (Figure `fig:preprocessing`),
Results, Section "Datasets", and the dataset tables of the appendix.

Output: `results/figures/preprocessing_zyla_static.*` and `results/figures/preprocessing_phantom_static.*`.

In [ ]:
%matplotlib inline
import numpy as np
import pandas as pd
from IPython.display import display

import thesis_config as C     # paths and settings
import thesis_ml as M         # data, preprocessing, models, training and evaluation

pd.set_option('display.precision', 3)
C.RESULTS_DIR.mkdir(parents=True, exist_ok=True)
print('Results folder:', C.RESULTS_DIR.resolve())
if C.FAST:
    print('QUICK TEST (THESIS_FAST=1): few images and 3 epochs, the numbers are not meaningful.')

In [ ]:
import matplotlib.pyplot as plt
import thesis_style as S
S.apply()

## 1. Loading
The output lists the size filter limits per dataset (shorter crop side, 1st to 99th percentile) and the number of
images after all exclusions. For the Andor Zyla flow dataset, the edge check reports how many crops reached beyond
the camera ROI.

In [ ]:
data = C.load_data(sets=['zstack', 'flow_zyla', 'phantom_static', 'flow_phantom', 'flow_apd'])

## 2. Number of images per dataset, imaging mode and cell line

In [ ]:
allm = pd.concat([d.meta.assign(dataset=k) for k, d in data.items()], ignore_index=True)
allm.loc[allm.dataset.isin(['zstack', 'flow_zyla']), 'mode'] = ''
counts = allm.pivot_table(index=['dataset', 'mode'], columns='label', values='uid', aggfunc='count', fill_value=0)
counts['total'] = counts.sum(axis=1)
counts['HeLa fraction'] = counts['HeLa'] / counts['total']
totals = allm.groupby('dataset').label.value_counts().unstack()
totals['HeLa fraction'] = totals['HeLa'] / totals.sum(axis=1)
display(counts.round(3))
display(totals.round(3))
display(allm.groupby('dataset').agg(images=('uid', 'size'), groups=('group', 'nunique')))

Macro-F1 of a classifier that assigns every image to the larger class: the larger class has an F1 score of
2p/(1 + p) and the smaller class 0, so the macro-F1 is p/(1 + p), with p the fraction of the larger class.

In [ ]:
p_major = totals[C.CLASSES].max(axis=1) / totals[C.CLASSES].sum(axis=1)
display((p_major / (1 + p_major)).rename('macro-F1, all images in the larger class').round(3))

### Z-stack: stacks and acquisition days
Pixel size per day: 0.325 µm/px without binning, 0.65 µm/px with 2 × 2 binning (`ZSTACK_PX_UM` in
`thesis_config.py`).

In [ ]:
zs = data['zstack'].meta
display(zs.groupby(['label', 'date']).agg(images=('uid', 'size'), stacks=('group_stack', 'nunique'),
                                         raw_px_um=('px_um', 'first')))

### Static Phantom: images and raw images per day, cell line and imaging mode

In [ ]:
ps = data['phantom_static'].meta
display(ps.groupby(['date', 'label', 'mode']).agg(images=('uid', 'size'), raw_images=(C.PS_GROUP_COL, 'nunique'))
        .unstack('mode'))
print('Groups (raw images over both modes) per cell line:', ps.groupby('label')[C.PS_GROUP_COL].nunique().to_dict())

### APD dataset: cells per video after the size filter

In [ ]:
apd = data['flow_apd'].meta
per_video = apd.groupby(['date', 'video', 'label']).size().unstack('label', fill_value=0)
per_video['total'] = per_video.sum(axis=1)
display(per_video)
display(per_video.sum().rename('total'))

## 3. Preprocessing steps
One HeLa and one NIH3T3 crop of the z-stack dataset and of the static Phantom dataset (Ph2). The first two columns are
drawn on the same scale in µm. Second column: background subtracted and resampled to the common pixel size of the
setup (crops that are already coarser are not resampled). Third column: base image after padding to a square,
resizing to 64 × 64, circular background mask and percentile normalization. Fourth column: 8 × 8 input of the chip
network.

The example cells are drawn at random with `SEED`. A specific cell can be fixed with `FIXED_EXAMPLES` (part of the
file name is enough). The z-stack examples are drawn from the images after the size filter, the static Phantom
examples from all crops of the selected imaging mode.

In [ ]:
SEED = 1
FIXED_EXAMPLES = {}            # e.g. {'zstack': {'HeLa': 'stack_raw_cell0141', 'NIH3T3': '...'}}
STATIC_MODE = 'Ph2'            # imaging mode of the static Phantom example
rng = np.random.default_rng(SEED)


def pick_zstack(label):
    meta = data['zstack'].meta
    idx = np.flatnonzero(meta.label.to_numpy() == label)
    fixed = FIXED_EXAMPLES.get('zstack', {}).get(label)
    hits = [i for i in idx if fixed and fixed in meta.path.iloc[i]]
    if hits:
        return meta.iloc[hits[0]]
    return meta.iloc[rng.choice(idx, size=1, replace=False)[0]]


def pick_static(label):
    fixed = FIXED_EXAMPLES.get('phantom_static', {}).get(label)
    raw = C.scan('phantom_static')                       # all crops, before the size filter
    sub = raw[raw['mode'].eq(STATIC_MODE) & raw.label.eq(label)]
    if fixed and sub.path.str.contains(fixed, regex=False).any():
        row = sub[sub.path.str.contains(fixed, regex=False)].iloc[0]
    else:
        row = sub.sample(1, random_state=SEED).iloc[0]
    trim = max(0, C.CROP_MARGIN_PX['phantom_static'] - int(round(C.MARGIN_UM / row.px_um)))
    row = row.copy()
    row['trim_px'] = trim
    return row


def no_axes(ax):
    ax.set_xticks([]); ax.set_yticks([])
    for sp in ax.spines.values():
        sp.set_visible(False)


def fig_steps(rows, P, name, caption):
    steps, ext = [], 0.0
    for lab, row in rows:
        st = M.preprocess_steps(row.path, row.px_um, P, int(row.trim_px))
        px_common = P.common_px_um if (P.common_px_um and row.px_um < P.common_px_um) else row.px_um
        steps.append((lab, row, st, px_common))
        ext = max(ext, *(np.array(st['raw'].shape) * row.px_um))
        print(f'{lab}: {row.path}')
    fig, axes = plt.subplots(len(rows), 4, figsize=S.figsize('full', aspect=len(rows) / 4 * 1.12), squeeze=False,
                             gridspec_kw=dict(wspace=0.12, hspace=0.35))
    for r, (lab, row, st, px_common) in enumerate(steps):
        for j, (key, px) in enumerate([('raw', row.px_um), ('common', px_common)]):
            im, ax = st[key], axes[r, j]
            h, w = np.array(im.shape) * px
            lo, hi = np.percentile(im, [1, 99])
            ax.imshow(im, cmap='gray', vmin=lo, vmax=hi,
                      extent=[(ext - w) / 2, (ext + w) / 2, (ext + h) / 2, (ext - h) / 2])
            ax.set_xlim(0, ext); ax.set_ylim(ext * 1.18, 0)
            ax.plot([ext * 0.05, ext * 0.05 + 10], [ext * 1.09] * 2, color=S.INK, lw=1.5, solid_capstyle='butt')
            ax.text(ext * 0.05 + 12, ext * 1.09, '10 µm', va='center', fontsize=plt.rcParams['xtick.labelsize'])
            ax.set_xlabel(f'{im.shape[1]} × {im.shape[0]} px\n{px:g} µm/px', fontsize=plt.rcParams['xtick.labelsize'])
        axes[r, 2].imshow(st['base'], cmap='gray', vmin=0, vmax=1)
        axes[r, 3].imshow(st['small'], cmap='gray', vmin=0, vmax=1)
        axes[r, 0].set_ylabel(lab, rotation=0, ha='right', va='center')
    for ax in axes.flat:
        no_axes(ax)
    for j, t in enumerate(['Crop\n(margin trimmed)', f'Resampled\nto {P.common_px_um:g} µm/px',
                           f'{P.base_size} × {P.base_size}\nbase image', '8 × 8\nchip input']):
        axes[0, j].set_title(t)
    S.save_figure(fig, name, caption, C.FIGURE_DIR)


TEXT = ('First column: crop after trimming the background margin to {m:g} µm. Second column: background subtracted, '
        'at {px:g} µm/px. Both columns are drawn on the same scale in µm. Third column: base image after padding to '
        'a square, resampling, circular background mask and percentile normalization. Fourth column: 8 × 8 input '
        'of the chip network.')
P = C.preproc_for('zstack')
fig_steps([(lab, pick_zstack(lab)) for lab in C.CLASSES], P, 'preprocessing_zyla_static',
          'Z-stack dataset: preprocessing of one HeLa and one NIH3T3 crop. ' + TEXT.format(m=C.MARGIN_UM,
                                                                                           px=P.common_px_um))
P = C.preproc_for('phantom_static')
fig_steps([(lab, pick_static(lab)) for lab in C.CLASSES], P, 'preprocessing_phantom_static',
          f'Static Phantom dataset ({STATIC_MODE}): preprocessing of one HeLa and one NIH3T3 crop. '
          + TEXT.format(m=C.MARGIN_UM, px=P.common_px_um))